In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/olympic_1896_2024_extended.csv")

print(df.shape)
df.head()


(274574, 7)


,Name,Sex,NOC,Year,Sport,Event,Medal
0,A Dijiang,M,CHN,1992,Basketball,Basketball Men's Basketball,NaN
1,A Lamusi,M,CHN,2012,Judo,Judo Men's Extra-Lightweight,NaN
2,Gunnar Nielsen Aaby,M,DEN,1920,Football,Football Men's Football,NaN
3,Edgar Lindenau Aabye,M,DEN,1900,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,Christine Jacoba Aaftink,F,NED,1988,Speed Skating,Speed Skating Women's 500 metres,NaN


In [2]:
india_df = df[df["NOC"] == "IND"].copy()

print("India-only rows:", india_df.shape)
india_df.head()


India-only rows: (1408, 7)


,Name,Sex,NOC,Year,Sport,Event,Medal
505,S. Abdul Hamid,M,IND,1928,Athletics,Athletics Men's 110 metres Hurdles,NaN
506,S. Abdul Hamid,M,IND,1928,Athletics,Athletics Men's 400 metres Hurdles,NaN
895,Shiny Kurisingal Abraham-Wilson,F,IND,1984,Athletics,Athletics Women's 800 metres,NaN
896,Shiny Kurisingal Abraham-Wilson,F,IND,1984,Athletics,Athletics Women's 4 x 400 metres Relay,NaN
897,Shiny Kurisingal Abraham-Wilson,F,IND,1988,Athletics,Athletics Women's 800 metres,NaN


In [3]:
india_df["has_medal"] = india_df["Medal"].notna().astype(int)


In [4]:
def recency_weight(year):
    if year <= 2012:
        return 1.0
    elif year == 2016:
        return 1.5
    elif year == 2020:
        return 2.0
    elif year == 2024:
        return 3.0

india_df["recency_weight"] = india_df["Year"].apply(recency_weight)


In [5]:
sport_features = (
    india_df
    .groupby("Sport")
    .agg(
        total_participations=("Year", "count"),
        total_medals=("has_medal", "sum"),
        weighted_medals=("has_medal", lambda x: np.sum(x * india_df.loc[x.index, "recency_weight"])),
        last_participation=("Year", "max"),
        olympics_participated=("Year", "nunique")
    )
    .reset_index()
)

sport_features.head()


,Sport,total_participations,total_medals,weighted_medals,last_participation,olympics_participated
0,Alpine Skiing,16,0,0.0,2014,7
1,Alpinism,7,7,7.0,1924,1
2,Archery,56,0,0.0,2016,7
3,Art Competitions,2,0,0.0,1948,1
4,Athletics,269,2,2.0,2016,24


In [6]:
sport_features["medal_rate"] = (
    sport_features["total_medals"] /
    sport_features["total_participations"].replace(0, np.nan)
)

sport_features["medal_rate"] = sport_features["medal_rate"].fillna(0)


In [7]:
sport_features.to_csv(
    "../data/processed/india_sport_features.csv",
    index=False
)

print("India sport-wise features saved")


India sport-wise features saved
